# 10. M7 crime exposure windows: 6, 12, 24 and 36 completed months

This notebook compares MPS crime measured over the previous 6, 12, 24 and 36 completed months. The sale month is excluded, and each window requires every contributing month to be observed. All specifications use the M7 controls, LSOA fixed effects, year-quarter fixed effects and LSOA-clustered standard errors.

**Comparison design:** The main four-window comparison re-estimates all twelve crime models on the same 234,070 sales from April 2013 onward. The notebook first retains intermediate 12-versus-24 and 12/24/36 tables to show the sample restrictions, and separately compares 6 versus 12 months on the full 338,347-sale M7 sample. The only coefficient figure shows all four windows on the common sample.

Each regressor is `log(1 + cumulative crime count)`. Longer windows change both the count scale and its remaining variation, so the tables also report the association for one standard deviation of crime after removing controls and fixed effects. These observational comparisons cannot establish delayed price adjustment or a causal effect of crime.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.linalg import qr
from linearmodels.iv import AbsorbingLS
from IPython.display import display

ROOT=Path.cwd()
DATA=ROOT/'data'
OUT=DATA/'10_crime_mps_exposure_windows_outputs'
OUT.mkdir(parents=True,exist_ok=True)
MONTHLY=DATA/'mps_monthly_panel.csv.gz'
MERGED=DATA/'crime_housing_merged.csv.gz'
BASELINE=DATA/'02_crime_mps_outputs'/'combined_primary_models.csv'
for path in [MONTHLY,MERGED,BASELINE]:
    if not path.exists(): raise FileNotFoundError(path)
MAIN=['mps_all','mps_violence','mps_property']
EXPOSURES=['annual','lagyear','prior12']
PROPERTY=['detached_dum','semi_d_dum','terrace_dum','freehold','newbuild']
LOCATION=['tube_distnear_km','bus_distnear_km','thamesriv_dist_km','dist_to_cbd','Nearest_Park_dist_km']
ECONOMIC=['grossannualpay_k','hoursworked','unemployment']
CONTROLS=PROPERTY+LOCATION+ECONOMIC
CRIME_COLS=['log1p_'+crime+'_'+exposure for exposure in EXPOSURES for crime in MAIN]
SOURCE_CRIME=[crime+'_'+exposure for exposure in EXPOSURES for crime in MAIN]

monthly=pd.read_csv(MONTHLY,usecols=['lsoacode','month']+MAIN,low_memory=False)
monthly['month']=pd.PeriodIndex(monthly.month,freq='M')
monthly=monthly.sort_values(['lsoacode','month']).reset_index(drop=True)
if monthly.duplicated(['lsoacode','month']).any():raise ValueError('Duplicate LSOA-month in MPS panel')
if (monthly.groupby('lsoacode').month.nunique()!=69).any():
    raise ValueError('Incomplete MPS month grid; inspect source coverage')
for crime in MAIN:
    monthly[crime+'_prior12_check']=monthly.groupby('lsoacode')[crime].transform(
        lambda s:s.shift(1).rolling(12,min_periods=12).sum())
    monthly[crime+'_prior24']=monthly.groupby('lsoacode')[crime].transform(
        lambda s:s.shift(1).rolling(24,min_periods=24).sum())
window=monthly[['lsoacode','month']+[c+'_prior12_check' for c in MAIN]+[c+'_prior24' for c in MAIN]]
window=window.rename(columns={'month':'sale_month'})
window.to_csv(OUT/'lsoa_month_prior24_exposures.csv.gz',index=False,compression='gzip')

raw_columns=['price','age','grossannualpay','Nearest_Park_dist','tube_distnear',
             'bus_distnear','thamesriv_dist','dist_to_cbd','detached_dum','semi_d_dum',
             'terrace_dum','freehold','hoursworked','unemployment','lsoacode',
             'msoa11','oslaua','year','yearquarter','sale_month','analysis_geography_ok',
             'mps_exposures_available']+SOURCE_CRIME
data=pd.read_csv(MERGED,usecols=raw_columns,low_memory=False)
data['sale_month']=pd.PeriodIndex(data.sale_month,freq='M')
data=data.merge(window,on=['lsoacode','sale_month'],how='left',validate='many_to_one')
for crime in MAIN:
    supplied=data[crime+'_prior12']
    checked=data[crime+'_prior12_check']
    if not np.array_equal(supplied.loc[supplied.notna()].to_numpy(),
                          checked.loc[supplied.notna()].to_numpy()):
        raise AssertionError(f'24-month construction does not reproduce prior12: {crime}')
    data['log1p_'+crime+'_prior24']=np.log1p(data[crime+'_prior24'])
data['lnprice']=np.log(data.price.where(data.price.gt(0)))
data['newbuild']=data.age.eq('Y').astype(float)
data['grossannualpay_k']=data.grossannualpay/1000
data['Nearest_Park_dist_km']=data.Nearest_Park_dist/1000
for source in ['tube_distnear','bus_distnear','thamesriv_dist']:
    data[source+'_km']=data[source]/1000
for source,target in zip(SOURCE_CRIME,CRIME_COLS):
    if data[source].dropna().lt(0).any():raise ValueError('Negative crime count: '+source)
    data[target]=np.log1p(data[source])
required=['lnprice','lsoacode','msoa11','oslaua','year','yearquarter']+CONTROLS+CRIME_COLS
valid=data[required].replace('',np.nan).notna().all(axis=1)
valid &= np.isfinite(data[required].select_dtypes(include='number')).all(axis=1)
valid &= data.analysis_geography_ok & data.mps_exposures_available
baseline=pd.read_csv(BASELINE).query("model == 'M7' and exposure_label == 'prior12'")
assert len(baseline)==3 and valid.sum()==int(baseline.N.iloc[0])
matched=valid & data[[crime+'_prior24' for crime in MAIN]].notna().all(axis=1)
sample=data.loc[matched,['lnprice','lsoacode','year','yearquarter']+CONTROLS+
                ['log1p_'+crime+'_prior12' for crime in MAIN]+
                ['log1p_'+crime+'_prior24' for crime in MAIN]].copy()
coverage=pd.DataFrame({'year':sorted(data.loc[valid,'year'].unique())})
coverage['baseline_prior12_sales']=coverage.year.map(data.loc[valid].groupby('year').size()).fillna(0).astype(int)
coverage['matched_prior12_prior24_sales']=coverage.year.map(sample.groupby('year').size()).fillna(0).astype(int)
coverage['retained_share']=coverage.matched_prior12_prior24_sales/coverage.baseline_prior12_sales
coverage.to_csv(OUT/'prior24_sample_coverage.csv',index=False)
display(coverage)
print(f'Original prior12 sample: {valid.sum():,} sales; matched 12/24 sample: {len(sample):,} sales.')
print('Matched sale months:',data.loc[matched,'sale_month'].min(),data.loc[matched,'sale_month'].max())

,year,baseline_prior12_sales,matched_prior12_prior24_sales,retained_share
0,2012,85375,64856,0.75966
1,2013,99429,99429,1.00000
2,2014,108207,108207,1.00000
3,2015,45336,45336,1.00000


Original prior12 sample: 338,347 sales; matched 12/24 sample: 317,828 sales.
Matched sale months: 2012-04 2015-07


## Same-sample M7 regressions

All six crime variables and the common controls are absorbed together under LSOA and year-quarter effects. The crime coefficient and LSOA-clustered standard error follow notebook 02's estimator. The partial crime SD is calculated after removing **both** fixed effects and the other controls; coefficient × partial SD is the estimated log-price difference associated with a one-SD difference in the remaining crime signal.

In [2]:
def absorb(matrix,groups,tol=1e-10,max_iter=2000):
    z=np.asarray(matrix,dtype=float).copy()
    codes=[pd.factorize(g,sort=False)[0] for g in groups]
    if any((c<0).any() for c in codes):raise ValueError('Missing FE identifier')
    for iteration in range(max_iter):
        before=z.copy()
        for g in codes:
            counts=np.bincount(g)
            sums=np.column_stack([np.bincount(g,weights=z[:,j],minlength=len(counts))
                                  for j in range(z.shape[1])])
            z-=(sums/counts[:,None])[g]
        if np.max(np.abs(z-before))<tol:return z,iteration+1
    raise RuntimeError('Fixed-effect absorption did not converge')

exposure_cols=['log1p_'+crime+'_'+window for crime in MAIN for window in ['prior12','prior24']]
columns=['lnprice']+exposure_cols+CONTROLS
matrix=sample[columns].to_numpy(float,copy=True)
scale=matrix.std(axis=0);scale[scale<1e-14]=1
projected,iterations=absorb(matrix/scale,[sample.lsoacode,sample.yearquarter])
projected[:,projected.std(axis=0)<1e-7]=0
prepared=pd.DataFrame(projected*scale,columns=columns,index=sample.index)

# Use the same control projection to compare the identifying crime variation.
control_matrix=prepared[CONTROLS].to_numpy(float)
Q,R,piv=qr(control_matrix,mode='economic',pivoting=True)
diagonal=np.abs(np.diag(R))
rank=int((diagonal>diagonal[0]*1e-10).sum())
Q=Q[:,:rank]
G=sample.lsoacode.nunique()
critical=stats.t.ppf(.975,G-1)

def fit_one(name):
    names=[name]+CONTROLS
    x=prepared[names].astype(float).copy()
    y=prepared.lnprice.astype(float).copy()
    scales=x.std(axis=0).to_numpy().copy()
    scales[scales<1e-14]=1
    x_scaled=x/scales
    x_scaled-=x_scaled.mean(axis=0)
    y-=y.mean()
    fit=AbsorbingLS(y,x_scaled,absorb=None,drop_absorbed=True).fit(
        cov_type='clustered',clusters=sample.lsoacode,debiased=True)
    if name not in fit.params.index:raise ValueError('Crime absorbed: '+name)
    beta=float(fit.params[name])/scales[0]
    se=float(fit.std_errors[name])/scales[0]
    crime_part=prepared[name].to_numpy()-Q@(Q.T@prepared[name].to_numpy())
    partial_sd=float(crime_part.std(ddof=1))
    return dict(N=len(sample),clusters=G,beta=beta,se=se,
                p=float(2*stats.t.sf(abs(beta/se),G-1)),
                ci_low=beta-critical*se,ci_high=beta+critical*se,
                crime_sd_after_fe=float(prepared[name].std()),
                crime_sd_after_fe_and_controls=partial_sd,
                beta_per_partial_crime_sd=beta*partial_sd,r2=fit.rsquared,
                fe_iterations=iterations)

rows=[]
for crime in MAIN:
    for window in ['prior12','prior24']:
        rows.append(dict(model='M7',crime=crime,exposure=window,
                         **fit_one('log1p_'+crime+'_'+window)))
results=pd.DataFrame(rows)
results.to_csv(OUT/'m7_prior12_vs_prior24_regressions.csv',index=False)
display(results[['crime','exposure','N','beta','se','p','ci_low','ci_high',
                 'crime_sd_after_fe_and_controls','beta_per_partial_crime_sd']])

,crime,exposure,N,beta,se,p,ci_low,ci_high,crime_sd_after_fe_and_controls,beta_per_partial_crime_sd
0,mps_all,prior12,317828,-0.000031,0.006424,0.996131,-0.012625,0.012563,0.127458,-0.000004
1,mps_all,prior24,317828,0.006733,0.009495,0.478301,-0.011882,0.025348,0.082045,0.000552
2,mps_violence,prior12,317828,0.002382,0.003085,0.440115,-0.003666,0.008429,0.245074,0.000584
3,mps_violence,prior24,317828,-0.006648,0.005221,0.203022,-0.016884,0.003589,0.156249,-0.001039
4,mps_property,prior12,317828,-0.001895,0.004907,0.699405,-0.011515,0.007725,0.151605,-0.000287
5,mps_property,prior24,317828,0.004882,0.007638,0.522783,-0.010093,0.019856,0.097658,0.000477


## Compare the windows

The coefficient comparison below is paired on the matched sample. `coefficient_change` is prior24 minus prior12. The original full-sample prior12 estimate is retained in the CSV to show the effect of losing early transactions; it is **not** used to judge the exposure-window difference.

In [3]:
comparison=[]
for crime in MAIN:
    old=baseline.loc[baseline.crime.eq(crime)].iloc[0]
    p12=results.loc[results.crime.eq(crime)&results.exposure.eq('prior12')].iloc[0]
    p24=results.loc[results.crime.eq(crime)&results.exposure.eq('prior24')].iloc[0]
    comparison.append(dict(crime=crime,baseline_prior12_N=int(old.N),matched_N=int(p12.N),
                           baseline_prior12_beta=old.beta,baseline_prior12_p=old.p,
                           matched_prior12_beta=p12.beta,matched_prior12_se=p12.se,
                           matched_prior12_p=p12.p,
                           matched_prior24_beta=p24.beta,matched_prior24_se=p24.se,
                           matched_prior24_p=p24.p,
                           coefficient_change=p24.beta-p12.beta,
                           matched_prior12_standardized=p12.beta_per_partial_crime_sd,
                           matched_prior24_standardized=p24.beta_per_partial_crime_sd,
                           standardized_change=p24.beta_per_partial_crime_sd-p12.beta_per_partial_crime_sd))
comparison=pd.DataFrame(comparison)
comparison.to_csv(OUT/'m7_prior24_comparison.csv',index=False)
display(comparison)

labels={'mps_all':'All crime','mps_violence':'Violent crime','mps_property':'Property crime'}

,crime,baseline_prior12_N,matched_N,baseline_prior12_beta,baseline_prior12_p,matched_prior12_beta,matched_prior12_se,matched_prior12_p,matched_prior24_beta,matched_prior24_se,matched_prior24_p,coefficient_change,matched_prior12_standardized,matched_prior24_standardized,standardized_change
0,mps_all,338347,317828,-0.000586,0.922912,-0.000031,0.006424,0.996131,0.006733,0.009495,0.478301,0.006764,-0.000004,0.000552,0.000556
1,mps_violence,338347,317828,0.000214,0.942549,0.002382,0.003085,0.440115,-0.006648,0.005221,0.203022,-0.009029,0.000584,-0.001039,-0.001622
2,mps_property,338347,317828,-0.000165,0.971914,-0.001895,0.004907,0.699405,0.004882,0.007638,0.522783,0.006776,-0.000287,0.000477,0.000764


## Additional 36-month check on a new common sample

The MPS monthly panel starts in April 2010. A complete previous-36-month count is first available for sales in April 2013. The next comparison re-estimates **all three windows** on those same sales. Its 12- and 24-month coefficients will therefore differ from the earlier 12-versus-24 comparison, which retains more sales. The earlier comparison remains above so the cost of each sample restriction is visible.

In [4]:
# Add the 36 completed months before the sale month; require every month.
for crime in MAIN:
    monthly[crime+'_prior36']=monthly.groupby('lsoacode')[crime].transform(
        lambda s:s.shift(1).rolling(36,min_periods=36).sum())
window36=monthly[['lsoacode','month']+[crime+'_prior36' for crime in MAIN]].rename(
    columns={'month':'sale_month'})
data36=data.merge(window36,on=['lsoacode','sale_month'],how='left',validate='many_to_one')
for crime in MAIN:
    data36['log1p_'+crime+'_prior36']=np.log1p(data36[crime+'_prior36'])
common36=valid & data36[[crime+'_prior36' for crime in MAIN]].notna().all(axis=1)
WINDOWS=['prior12','prior24','prior36']
sample=data36.loc[common36,['lnprice','lsoacode','year','yearquarter']+CONTROLS+
                  ['log1p_'+crime+'_'+window for crime in MAIN for window in WINDOWS]].copy()
coverage36=pd.DataFrame({'year':sorted(data36.loc[valid,'year'].unique())})
coverage36['full_prior12_sales']=coverage36.year.map(data36.loc[valid].groupby('year').size()).fillna(0).astype(int)
coverage36['common_12_24_36_sales']=coverage36.year.map(sample.groupby('year').size()).fillna(0).astype(int)
coverage36.to_csv(OUT/'prior36_sample_coverage.csv',index=False)
display(coverage36)
print('Common 12/24/36 sales:',len(sample),'from',data36.loc[common36,'sale_month'].min(),
      'through',data36.loc[common36,'sale_month'].max())

# Rebuild the partialled-out design for this smaller, identical-sales sample.
exposure_cols=['log1p_'+crime+'_'+window for crime in MAIN for window in WINDOWS]
columns=['lnprice']+exposure_cols+CONTROLS
matrix=sample[columns].to_numpy(float,copy=True)
scale=matrix.std(axis=0);scale[scale<1e-14]=1
projected,iterations=absorb(matrix/scale,[sample.lsoacode,sample.yearquarter])
projected[:,projected.std(axis=0)<1e-7]=0
prepared=pd.DataFrame(projected*scale,columns=columns,index=sample.index)
control_matrix=prepared[CONTROLS].to_numpy(float)
Q,R,piv=qr(control_matrix,mode='economic',pivoting=True)
diagonal=np.abs(np.diag(R));rank=int((diagonal>diagonal[0]*1e-10).sum())
Q=Q[:,:rank]
G=sample.lsoacode.nunique();critical=stats.t.ppf(.975,G-1)

rows36=[]
for crime in MAIN:
    for window in WINDOWS:
        rows36.append(dict(model='M7',crime=crime,exposure=window,
                           **fit_one('log1p_'+crime+'_'+window)))
results36=pd.DataFrame(rows36)
assert results36.N.nunique()==1 and int(results36.N.iloc[0])==len(sample)
results36.to_csv(OUT/'m7_prior12_prior24_prior36_common_sample.csv',index=False)
display(results36[['crime','exposure','N','clusters','beta','se','p','ci_low','ci_high',
                   'crime_sd_after_fe_and_controls','beta_per_partial_crime_sd']])

,year,full_prior12_sales,common_12_24_36_sales
0,2012,85375,0
1,2013,99429,80527
2,2014,108207,108207
3,2015,45336,45336


Common 12/24/36 sales: 234070 from 2013-04 through 2015-07


,crime,exposure,N,clusters,beta,se,p,ci_low,ci_high,crime_sd_after_fe_and_controls,beta_per_partial_crime_sd
0,mps_all,prior12,234070,4647,0.000791,0.006651,0.905378,-0.012248,0.013830,0.114968,0.000091
1,mps_all,prior24,234070,4647,-0.001742,0.011330,0.877775,-0.023954,0.020469,0.068810,-0.000120
2,mps_all,prior36,234070,4647,-0.023078,0.016607,0.164714,-0.055636,0.009481,0.047605,-0.001099
3,mps_violence,prior12,234070,4647,0.003089,0.003536,0.382309,-0.003842,0.010021,0.223586,0.000691
4,mps_violence,prior24,234070,4647,-0.008865,0.005752,0.123364,-0.020142,0.002412,0.132361,-0.001173
5,mps_violence,prior36,234070,4647,-0.010604,0.008297,0.201269,-0.026869,0.005661,0.092798,-0.000984
6,mps_property,prior12,234070,4647,-0.004210,0.005426,0.437846,-0.014847,0.006427,0.137378,-0.000578
7,mps_property,prior24,234070,4647,-0.008779,0.009336,0.347131,-0.027082,0.009525,0.081744,-0.000718
8,mps_property,prior36,234070,4647,-0.034203,0.013562,0.011699,-0.060790,-0.007616,0.056494,-0.001932


## Findings from the three-window common sample

All nine estimates use **234,070 sales** from April 2013 onward and the same 4,647 LSOA clusters. The 36-month requirement excludes all 2012 and January–March 2013 transactions.

| Crime type | Prior 12 months: β (p) | Prior 24 months: β (p) | Prior 36 months: β (p) |
|---|---:|---:|---:|
| All crime | +0.00079 (0.905) | −0.00174 (0.878) | −0.02308 (0.165) |
| Violent crime | +0.00309 (0.382) | −0.00886 (0.123) | −0.01060 (0.201) |
| Property crime | −0.00421 (0.438) | −0.00878 (0.347) | −0.03420 (0.012) |

- The violent-crime point estimate becomes more negative as the window lengthens, but **its p-value does not continue to fall**: it rises from 0.123 at 24 months to 0.201 at 36 months. Its 36-month 95% confidence interval includes zero [−0.02687, +0.00566].
- Property crime becomes more negative and is statistically distinguishable from zero at 36 months (95% CI [−0.06079, −0.00762]). All crime remains imprecise at all three windows.
- The fully residualised log-crime SD falls sharply as the window lengthens. For violence it is 0.224, 0.132 and 0.093 at 12, 24 and 36 months. The corresponding coefficient × SD is +0.00069, −0.00117 and −0.00098. Thus the larger 36-month raw violent coefficient does **not** imply a larger association for a one-SD change in its remaining variation.
- These are exploratory window comparisons. The significant 36-month property result is worth reporting as a sensitivity finding, with the post-April-2013 sample restriction and the fact that several crime types and windows were examined. Separate coefficient p-values do not test whether two windows' coefficients differ. None of these results establishes delayed price adjustment or a causal effect of crime.


## Final short-window check: previous six completed months

A six-month window tests whether more recent crime is more closely associated with transaction prices. First, six and twelve months are compared on the full M7 prior-12 sample. Then all four windows are estimated on the **same** post-April-2013 sales used for the 36-month check. The four-window panel is the appropriate comparison of window trends; it sacrifices early sales solely to make the windows comparable.

In [5]:
# The sale month is excluded, and all six preceding monthly observations are required.
for crime in MAIN:
    monthly[crime+'_prior6']=monthly.groupby('lsoacode')[crime].transform(
        lambda s:s.shift(1).rolling(6,min_periods=6).sum())
window6=monthly[['lsoacode','month']+[crime+'_prior6' for crime in MAIN]].rename(
    columns={'month':'sale_month'})
data6=data36.merge(window6,on=['lsoacode','sale_month'],how='left',validate='many_to_one')
assert len(data6)==len(data36) and data6.index.equals(data36.index)
for crime in MAIN:
    data6['log1p_'+crime+'_prior6']=np.log1p(data6[crime+'_prior6'])
assert data6.loc[valid,[crime+'_prior6' for crime in MAIN]].notna().all().all()

def run_common_window_sample(mask,windows,path):
    global sample,prepared,Q,G,critical,iterations
    sample=data6.loc[mask,['lnprice','lsoacode','year','yearquarter']+CONTROLS+
                     ['log1p_'+crime+'_'+window for crime in MAIN for window in windows]].copy()
    exposure_cols=['log1p_'+crime+'_'+window for crime in MAIN for window in windows]
    columns=['lnprice']+exposure_cols+CONTROLS
    matrix=sample[columns].to_numpy(float,copy=True)
    scale=matrix.std(axis=0);scale[scale<1e-14]=1
    projected,iterations=absorb(matrix/scale,[sample.lsoacode,sample.yearquarter])
    projected[:,projected.std(axis=0)<1e-7]=0
    prepared=pd.DataFrame(projected*scale,columns=columns,index=sample.index)
    control_matrix=prepared[CONTROLS].to_numpy(float)
    Q,R,piv=qr(control_matrix,mode='economic',pivoting=True)
    diagonal=np.abs(np.diag(R));rank=int((diagonal>diagonal[0]*1e-10).sum())
    Q=Q[:,:rank]
    G=sample.lsoacode.nunique();critical=stats.t.ppf(.975,G-1)
    rows=[]
    for crime in MAIN:
        for window in windows:
            rows.append(dict(model='M7',crime=crime,exposure=window,
                             **fit_one('log1p_'+crime+'_'+window)))
    result=pd.DataFrame(rows)
    assert result.N.nunique()==1 and int(result.N.iloc[0])==len(sample)
    result.to_csv(OUT/path,index=False)
    return result

results6full=run_common_window_sample(valid,['prior6','prior12'],
    'm7_prior6_vs_prior12_full_sample.csv')
assert int(results6full.N.iloc[0])==int(baseline.N.iloc[0])
display(results6full[['crime','exposure','N','beta','se','p','ci_low','ci_high',
                      'crime_sd_after_fe_and_controls','beta_per_partial_crime_sd']])

FOUR_WINDOWS=['prior6','prior12','prior24','prior36']
results4=run_common_window_sample(common36,FOUR_WINDOWS,
    'm7_prior6_prior12_prior24_prior36_common_sample.csv')
assert int(results4.N.iloc[0])==int(results36.N.iloc[0])
for crime in MAIN:
    for window in ['prior12','prior24','prior36']:
        earlier=results36.loc[results36.crime.eq(crime)&results36.exposure.eq(window)].iloc[0]
        now=results4.loc[results4.crime.eq(crime)&results4.exposure.eq(window)].iloc[0]
        assert np.isclose(earlier.beta,now.beta,rtol=0,atol=1e-9)
display(results4[['crime','exposure','N','beta','se','p','ci_low','ci_high',
                  'crime_sd_after_fe_and_controls','beta_per_partial_crime_sd']])

fig,axes=plt.subplots(1,3,figsize=(16,4.7))
colours=['#7657a3','#2468a0','#d07824','#2c8a6e']
for ax,crime in zip(axes,MAIN):
    fitted=results4.loc[results4.crime.eq(crime)].set_index('exposure').loc[FOUR_WINDOWS]
    for j,(window,colour) in enumerate(zip(FOUR_WINDOWS,colours)):
        row=fitted.loc[window]
        ax.errorbar(j,row.beta,yerr=[[row.beta-row.ci_low],[row.ci_high-row.beta]],
                    fmt='o',color=colour,capsize=4,markersize=7)
        ax.annotate(f'{row.beta:+.4f}',(j,row.beta),xytext=(0,10),
                    textcoords='offset points',ha='center',fontsize=9)
    ax.axhline(0,color='#555',linestyle='--',linewidth=.8)
    ax.set_xticks(range(4),['6 months','12 months','24 months','36 months'])
    ax.set_xlim(-.5,3.5)
    ax.set_title(labels[crime]);ax.set_ylabel('Crime coefficient (95% clustered CI)')
    ax.grid(axis='y',alpha=.2)
fig.suptitle('M7 on identical sales: 6 to 36 completed months of MPS crime',fontsize=15)
fig.tight_layout()
fig.savefig(OUT/'m7_prior6_prior12_prior24_prior36_common_sample.png',dpi=170,bbox_inches='tight')
plt.close(fig)

,crime,exposure,N,beta,se,p,ci_low,ci_high,crime_sd_after_fe_and_controls,beta_per_partial_crime_sd
0,mps_all,prior6,338347,-0.002787,0.004228,0.509803,-0.011076,0.005502,0.184948,-0.000515
1,mps_all,prior12,338347,-0.000586,0.006055,0.922912,-0.012457,0.011285,0.129654,-0.000076
2,mps_violence,prior6,338347,0.001253,0.002297,0.585569,-0.003251,0.005756,0.349205,0.000437
3,mps_violence,prior12,338347,0.000214,0.002976,0.942549,-0.005619,0.006048,0.249921,0.000054
4,mps_property,prior6,338347,-0.001746,0.003012,0.562074,-0.007651,0.004159,0.220857,-0.000386
5,mps_property,prior12,338347,-0.000165,0.004693,0.971914,-0.009365,0.009035,0.154339,-0.000026


,crime,exposure,N,beta,se,p,ci_low,ci_high,crime_sd_after_fe_and_controls,beta_per_partial_crime_sd
0,mps_all,prior6,234070,-0.005606,0.004540,0.216943,-0.014507,0.003294,0.173719,-0.000974
1,mps_all,prior12,234070,0.000791,0.006651,0.905378,-0.012248,0.013830,0.114968,0.000091
2,mps_all,prior24,234070,-0.001742,0.011330,0.877775,-0.023954,0.020469,0.068810,-0.000120
3,mps_all,prior36,234070,-0.023078,0.016607,0.164714,-0.055636,0.009481,0.047605,-0.001099
4,mps_violence,prior6,234070,0.001585,0.002549,0.533924,-0.003411,0.006582,0.326720,0.000518
5,mps_violence,prior12,234070,0.003089,0.003536,0.382309,-0.003842,0.010021,0.223586,0.000691
6,mps_violence,prior24,234070,-0.008865,0.005752,0.123364,-0.020142,0.002412,0.132361,-0.001173
7,mps_violence,prior36,234070,-0.010604,0.008297,0.201269,-0.026869,0.005661,0.092798,-0.000984
8,mps_property,prior6,234070,-0.006370,0.003522,0.070601,-0.013276,0.000535,0.208940,-0.001331
9,mps_property,prior12,234070,-0.004210,0.005426,0.437846,-0.014847,0.006427,0.137378,-0.000578


## Final interpretation: six-month check

- **Full M7 sale sample (338,347 sales):** six-month all-crime β = −0.00279 (p = 0.510), violent-crime β = +0.00125 (p = 0.586), and property-crime β = −0.00175 (p = 0.562). None has a 95% confidence interval excluding zero. The paired 12-month estimates also remain close to zero.
- **Four-window common sample (234,070 sales from April 2013):** six-month all-crime β = −0.00561 (p = 0.217), violent-crime β = +0.00159 (p = 0.534), and property-crime β = −0.00637 (p = 0.071). The 24-month violent coefficient is −0.00886 (p = 0.123); its 36-month coefficient is −0.01060 (p = 0.201). Property crime reaches β = −0.03420 (p = 0.012) at 36 months.
- Therefore the six-month measure does **not** give convincing evidence that very recent crime is more strongly priced under M7. The property-crime six-month estimate differs noticeably between the full and restricted samples, underscoring sample sensitivity.
- Raw log-count coefficients change scale as the exposure window lengthens. The table reports coefficient × standard deviation of the crime regressor remaining after controls and fixed effects. Separate p-values test each coefficient against zero, not whether windows differ. Results from multiple exploratory windows and crime types should not be presented as a selected causal result.
